In [ ]:
from __future__ import print_function
import sys
import os
import subprocess
import fnmatch

import pandas as pd
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

from analysis_village.cc1pi.DataFrameUtils.DFLoading import *

In [ ]:
keys2load = ["cc1pi", "hdr", "histpotdf"] ## keys from the configuration file

file = "/exp/sbnd/data/users/lpelegri/cafpyana_data/old/cc1pi_1e20_training.df"
print(f"keys in {file}")
splh.print_keys(file)

## Check split multiplicity
print("n_split: %d" %splh.get_n_split(file))

## Define keys to load
print('dataframes')
 ## for big files, each key could have more than one split
training_1e20_df = splh.load_dfs(file, keys2load, 100)
print('loaded!')

training_1e20_df_evt_df = training_1e20_df["cc1pi"]
training_1e20_df_hdr_df = training_1e20_df["hdr"]

In [ ]:
import pandas as pd

def prune_dataset_by_event_list(df_dict, event_list_df, keys_to_process):
    if 'hdr' not in df_dict:
        print("Error: 'hdr' table must be present to perform coordinate matching.")
        return df_dict

    # --- STEP 1 & 2: Get the internal pointers from HDR ---
    # We reset the index to ensure 'run', 'subrun', 'evt', '__ntuple', and 'entry' 
    # are all available as columns for the comparison.
    hdr_reset = df_dict['hdr'].reset_index()

    # Create MultiIndices for the "Match"
    # This identifies rows where the physics coordinates exist in your blacklist
    blacklist_coords = pd.MultiIndex.from_frame(event_list_df[['run', 'subrun', 'evt']])
    hdr_coords = pd.MultiIndex.from_frame(hdr_reset[['run', 'subrun', 'evt']])

    # Find which rows in HDR are in the blacklist
    to_remove_mask = hdr_coords.isin(blacklist_coords)
    
    # Extract the unique internal pointers (__ntuple, entry) for those specific events
    # We use these because 'tracks' or 'showers' might not have 'run'/'evt' columns,
    # but they ALWAYS have these index levels.
    bad_pointers_df = hdr_reset.loc[to_remove_mask, ['__ntuple', 'entry']]
    blacklist_pointers = pd.MultiIndex.from_frame(bad_pointers_df)

    if len(blacklist_pointers) == 0:
        print("Optimization: No matching events from event_list found in this hdr table.")
        return df_dict, blacklist_pointers

    print(f"Found {len(blacklist_pointers)} events in 'hdr' to be filtered out.")

    # --- STEP 3: Filter those pointers out of all tables ---
    for key in keys_to_process:
        if key in df_dict and df_dict[key] is not None:
            curr_df = df_dict[key]
            
            # Extract the index levels from the current table to compare against our blacklist
            # This handles tables regardless of their specific MultiIndex depth
            curr_pointers_df = curr_df.index.to_frame(index=False)[['__ntuple', 'entry']]
            curr_pointers_idx = pd.MultiIndex.from_frame(curr_pointers_df)
            
            # The mask: Keep the row if its (__ntuple, entry) is NOT in the blacklist
            keep_mask = ~curr_pointers_idx.isin(blacklist_pointers)
            
            original_rows = len(curr_df)
            df_dict[key] = curr_df[keep_mask]
            
            print(f"  - {key}: Removed {original_rows - len(df_dict[key])} rows.")

    return df_dict, blacklist_pointers

In [ ]:

from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
event_list = training_1e20_df_hdr_df.reset_index()[['run', 'subrun', 'evt']].drop_duplicates()
event_list = event_list.sort_values(by=['run', 'subrun', 'evt'])
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file

n_split = 5
df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/mc_ar23p_extended_syst.df", keys2load, n_split)
pre_prune_evt_df = df['cc1pi']
pre_prune_hdr_df = df['hdr']

df, blacklist_pointers = prune_dataset_by_event_list(df, event_list, keys2load)

In [ ]:
mc_tot_pot = df['hdr']['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))

In [ ]:
import pandas as pd
import pathlib

def save_pruned_df(df_dict, output_path):
    """
    Saves the pruned dictionary of DataFrames to an HDF5 file 
    using the standard format.
    """
    out_file = pathlib.Path(output_path)
    out_file.parent.mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"SAVING PRUNED DATA TO: {out_file}")
    print(f"{'='*60}")

    with pd.HDFStore(out_file, mode='w') as hdf_out:
        for key, df_table in df_dict.items():
            if df_table is not None:
                # We save with the suffix _0 to match the split_0 convention
                hdf_key = f"{key}_0"
                print(f"Writing {key} ({len(df_table)} rows)...")
                hdf_out.put(key=hdf_key, value=df_table, format="fixed")
        
        # Add the 'split' metadata so load_df knows there is 1 split
        hdf_out.put(key="split", value=pd.DataFrame({"n_split": [1]}), format="fixed")

    print(f"\nDone! File saved successfully.")

In [ ]:
output_path = "/exp/sbnd/data/users/lpelegri/cafpyana_data/mc_ar23p_extended_syst_pruned.df"
if n_split != 100:
    output_path = "/exp/sbnd/data/users/lpelegri/cafpyana_data/mc_ar23p_extended_syst_pruned_low_stats.df" 
save_pruned_df(df, output_path)

# Double check that the evts match

In [ ]:
# 1. Align the pre_prune index to our blacklist pointers
pre_idx_df = pre_prune_evt_df.index.to_frame(index=False)[['__ntuple', 'entry']]
pre_multi_idx = pd.MultiIndex.from_frame(pre_idx_df)

# 2. Create the boolean mask for discarded events
# This returns a boolean array
discard_mask = pre_multi_idx.isin(blacklist_pointers)

# 3. Pull the nu_scores for these discarded slices
# We pass the mask directly to .loc
discarded_scores = pre_prune_evt_df.loc[discard_mask, [('slc', 'nu_score','','','','')]]

print(f"--- Discarded nu_scores (Total Found: {len(discarded_scores)}) ---")

if not discarded_scores.empty:
    # Print the first 20 to verify the values
    print(discarded_scores.head(20).to_string())
else:
    print("No discarded scores found. This suggests the blacklist_pointers do not overlap with this DataFrame.")

    

In [ ]:
# 1. Extract pointers from the pre_prune_hdr_df index
hdr_idx_df = pre_prune_hdr_df.index.to_frame(index=False)[['__ntuple', 'entry']]
hdr_multi_idx = pd.MultiIndex.from_frame(hdr_idx_df)

# 2. Create the mask based on your blacklist
discard_mask_hdr = hdr_multi_idx.isin(blacklist_pointers)

# 3. Select the physics ID columns for the discarded entries
# We use .copy() to avoid any SettingWithCopy warnings
discarded_ids = pre_prune_hdr_df.loc[discard_mask_hdr, ['run', 'subrun', 'evt']].copy()

print(f"--- Discarded Event IDs (Total: {len(discarded_ids)}) ---")

if not discarded_ids.empty:
    # Sort by run/subrun/evt so the list is readable
    print(discarded_ids.head(20).to_string())
else:
    print("No matching headers found for the provided blacklist_pointers.")

In [ ]:
# 1. Reset index of the training header
train_hdr_reset = training_1e20_df_hdr_df.reset_index()

# 2. Create MultiIndices for the physics coordinates
discarded_coords = pd.MultiIndex.from_frame(discarded_ids[['run', 'subrun', 'evt']])
train_coords = pd.MultiIndex.from_frame(train_hdr_reset[['run', 'subrun', 'evt']])

# 3. Filter for matches AND restrict to Run 2221
match_mask = train_coords.isin(discarded_coords)
training_matches = train_hdr_reset.loc[match_mask, ['run', 'subrun', 'evt', '__ntuple', 'entry']]

# 4. Filter specifically for Run 2221
run_2221_matches = training_matches[training_matches['run'] == 2221]

print(f"--- Training DF Pointers for Discarded IDs (Run 2221 Only) ---")
if not run_2221_matches.empty:
    # Sort by subrun and event for a clean list
    print(run_2221_matches.sort_values(['subrun', 'evt']).to_string(index=False))
    print(f"\nTotal discarded events found in Run 2221: {len(run_2221_matches)}")
else:
    print("No matches found for Run 2221. Either this run wasn't in the blacklist or it's not in this training file.")


In [ ]:
# 1. Get the specific pointers for Run 2221 from our previous match
run_2221_pointers_df = run_2221_matches[['__ntuple', 'entry']]
run_2221_pointers_idx = pd.MultiIndex.from_frame(run_2221_pointers_df)

# 2. Extract the index from the training cc1pi dataframe
# We align our search to the '__ntuple' and 'entry' levels
train_evt_idx_frame = training_1e20_df_evt_df.index.to_frame(index=False)[['__ntuple', 'entry']]
train_evt_multi_idx = pd.MultiIndex.from_frame(train_evt_idx_frame)

# 3. Filter the cc1pi dataframe
# REMOVED .values to avoid the AttributeError
final_check_mask = train_evt_multi_idx.isin(run_2221_pointers_idx)

# 4. Pull the scores
# We use the boolean array directly. 
# Also using a list for the column to keep it as a DataFrame for pretty printing.
target_col = [('slc', 'nu_score','','','','')]
final_check_df = training_1e20_df_evt_df.loc[final_check_mask, target_col]

print(f"--- Training nu_scores for Run 2221 Discarded Events ---")
if not final_check_df.empty:
    # Sorting by index so you can see them in order of entry/ntuple
    print(final_check_df.sort_index().to_string())
    
    # Calculate stats if possible
    avg_score = final_check_df[target_col[0]].mean()
    print(f"\nMean nu_score for these {len(final_check_df)} slices: {avg_score:.4f}")
else:
    print("No entries found in the cc1pi table for these specific pointers.")
    print("Check if the event was filtered out of the cc1pi table earlier in your selection.")